# 07 — True daily ranker lab

Purpose: prepare the data contract for a real daily cross-sectional ranker. This notebook does not treat a rank transform of regression scores as a ranker. It builds daily rank targets and daily group sizes from raw 10D returns.


In [ ]:
import sys, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init
from src.research.daily_ranker import prepare_ranker_frame
from src.research.notebook_lab_contracts import ResearchSessionConfig, CANONICAL_10D_RETURN_EXPR
from src.research.notebook_research_api import sanitize_factor_name

cfg_path = ROOT / "data" / "session_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    MARKET, SYMBOLS, BENCHMARK = cfg["market"], cfg["symbols"], cfg["benchmark"]
    TRAIN_START, TRAIN_END = cfg["train_start"], cfg["train_end"]
    TEST_START, TEST_END = cfg["test_start"], cfg["test_end"]
else:
    MARKET = "us"
    SYMBOLS = ["AAPL", "NVDA", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "AVGO", "COST", "NFLX"]
    BENCHMARK = "QQQ"
    TRAIN_START, TRAIN_END = "2021-01-01", "2024-12-31"
    TEST_START, TEST_END = "2025-01-01", "2026-06-18"

config = ResearchSessionConfig(
    market=MARKET,
    symbols=SYMBOLS,
    benchmark=BENCHMARK,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
    experiment_id=f"{MARKET}_true_daily_ranker_lab",
    return_expression=CANONICAL_10D_RETURN_EXPR,
)

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D

print(config)


In [ ]:
feature_exprs = [
    "$close/Ref($close,5)-1",
    "$close/Ref($close,10)-1",
    "$close/Ref($close,20)-1",
    "Std($ret,10)",
    "$volume/Ref($volume,10)-1",
]
features = D.features(SYMBOLS, feature_exprs, start_time=TRAIN_START, end_time=TRAIN_END)
raw = D.features(SYMBOLS, [config.return_expression], start_time=TRAIN_START, end_time=TRAIN_END)

for frame in (features, raw):
    if frame.index.names == ["instrument", "datetime"]:
        frame.index = frame.index.swaplevel()
        frame.sort_index(inplace=True)

features = features.fillna(0.0).replace([np.inf, -np.inf], 0.0)
features.columns = [sanitize_factor_name(expr) for expr in feature_exprs]
raw_returns = raw.copy()
raw_returns.columns = ["return"]
raw_returns.attrs["provenance"] = "raw_forward_return"
raw_returns.attrs["horizon"] = 10

X_rank, y_rank, groups = prepare_ranker_frame(features, raw_returns)

print(f"ranker features: {X_rank.shape}")
print(f"rank target: {y_rank.shape}")
print(f"groups: {len(groups)} dates, first five sizes={groups[:5]}")
print(y_rank.describe())


## Next step

The next PR can add a real LightGBM/XGBoost ranking objective using `X_rank`, `y_rank`, and `groups`, then evaluate out-of-sample prediction scores through `run_10d_experiment(...)` against raw 10D returns.
